# Config

In [1]:
!pip install nltk==3.9.1
!pip install mlflow==3.3.1

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [3]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json

# 1) Preprocesamiento de los datos


In [6]:
# 1) Cargar datos
path = "/tmp/data"
path_analytics = "/tmp/analytics"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")

# 2) Guardar qué secuencias de palabras del resumen serán eliminadas al aplicar get_expressions_to_delete()
list_texts = df["Resumen"].to_list()
df_deleted = check_deleted_expressions(list_texts)
savepath=os.path.join(path_analytics, "deleted_re.xlsx")
df_deleted.to_excel(savepath, index=False)

# 3) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
savepath=os.path.join(path, "data_clean.xlsx")
df.to_excel(savepath, index=False)

# 2) Traducción del texto

In [ ]:
#Crear columna de registro de idioma: 
# True: Texto en español, False: Texto en inglés
df["Español"]=detect_language(df["Resumen_trad"])

In [9]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()


#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.xlsx")
df.to_excel(savepath, index=False)

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Usando dispositivo: cuda


model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Traduciendo: 100%|██████████| 119/119 [00:25<00:00,  4.64batch/s]


Tiempo total de traducción: 755.99 segundos


In [10]:
#3. Selección de columnas que se utilizarán en clasificador y concatenación
# Última limpieza antes de generar concatenación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad", "Facultad_del_Proyecto_trad", "Depto_Persona_trad"]
for col in cols:
    df[col] = df[col].apply(final_clean)

#  Selección de columnas para embedding.
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
element_names = ["title", "keywords", "abstract"]
df = gen_text_for_embedding(df, cols, element_names)

# Guardado de resultados
savepath=os.path.join(path, "data_translated_concat.xlsx")
df.to_excel(savepath, index=False)
savepath=os.path.join(path, "data_translated_concat.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

df.head()

,Código VRID,Interdisciplinario,Transdisciplinario,Título,Keywords,Resumen,Facultad del Proyecto,Depto Persona,Titulo_trad,Resumen_trad,keywords_trad,Facultad_del_Proyecto_trad,Depto_Persona_trad,Español,text_for_embedding_translated
0,217.173.049-1.0,SI,,PATRONES DE CRIANZA Y SOCIALIZACIÓN DE GÉNERO ...,,OBJETIVOS GENERALES: _x000D_\nDESCRIBIR LOS PR...,FACULTAD DE CIENCIAS SOCIALES,"DEPARTAMENTO DE ECONOMÍA, SIN INFORMACIÓN, ESC...",patterns of gender upbringing and socializatio...,general objectives: to describe the processes ...,,faculty of social sciences,"department of economics, without information, ...",True,title: patterns of gender upbringing and socia...
1,218.201.002-1.0,SI,,ADAPTACIÓN CULTURAL Y VALIDACIÓN DE LA ESCALA ...,"ESTILO DE VIDA, ADOLESCENTES _x000D_\n",PARA EVALUAR LOS COMPORTAMIENTOS RELACIONADOS ...,FACULTAD DE ENFERMERÍA,"DEPARTAMENTO DE CIENCIA ANIMAL, DEPARTAMENTO D...",cultural adaptation and validation of the life...,in order to evaluate the behaviors related to ...,"lifestyle, teens",faculty of nursing,"department of animal science, department of pl...",True,title: cultural adaptation and validation of t...
2,218.102.031-1.0IN,NO,,PROMOVIENDO LA REFLEXIÓN EN ESTUDIANTES DE PRE...,,EL PRESENTE PROYECTO INVOLUCRA LA REALIZACIÓN ...,FACULTAD DE ODONTOLOGÍA,DEPARTAMENTO DE ASTRONOMÍA,promoting reflection in preclinical dental stu...,the present project involves the realization o...,,faculty of dentistry,department of astronomy,True,title: promoting reflection in preclinical den...
3,218.163.016-INI,INDEFINIDO,,MOTIVACIÓN Y HABILIDADES SOCIALES EN ADOLESCENTES,,EL ESTUDIO DE LA MOTIVACIÓN TIENE DIFERENTES A...,FACULTAD DE EDUCACIÓN,"DEPARTAMENTO DE CIENCIAS DE LA EDUCACIÓN, DEPT...",motivation and social skills in adolescents,the study of the motivation has different side...,,faculty of education,"department of education sciences, department o...",True,title: motivation and social skills in adolesc...
4,219.091.052-INI,NO,,TIME EFFECTS ON THE LIQUEFACTION RESPONSE OF G...,,SECONDARY CONSOLIDATION AND AGEING ARE TWO OFT...,FACULTAD DE INGENIERÍA,DEPTO. TEORÍA POLITICA Y FUND.DE LA EDUC.,time effects on the liquefaction response of g...,secondary consolidation and ageing are two oft...,,faculty of engineering,department of political and fund theory of edu...,False,title: time effects on the liquefaction respon...


# 3) Split dataset

In [11]:
def to_serializable(obj):
    if hasattr(obj, "tolist"):
        return obj.tolist()
    return obj

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
import json

path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

df = df[df["Interdisciplinario"] != "INDEFINIDO"]
le = LabelEncoder()
df["labels"] = le.fit_transform(df["Interdisciplinario"])

# Convertir a arrays
ids = df["Código VRID"].to_numpy()
labels = df["labels"].to_numpy()

# Train/Test split (ids y labels en paralelo)
idx_train, idx_test, y_train, y_test = train_test_split(
    ids,
    labels,
    test_size=0.2,
    random_state=7,
    stratify=labels
)

# Crear folds sobre train
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

folds = []
for fold, (train_pos, val_pos) in enumerate(skf.split(idx_train, y_train)):
    train_ids = idx_train[train_pos]   # array de IDs
    val_ids = idx_train[val_pos]       # array de IDs
    folds.append(val_ids)

print("Test size:", len(idx_test))
print("Fold 0 - Val size:", len(folds[0]))

#Guardar index en diccionario
dataset_index = {
    "Train": idx_train,
    "Test": idx_test,
    "kfolds": folds 
}
filepath=os.path.join(path, "train_test_ids_3folds.json")

# Guardar
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)


Test size: 193
Fold 0 - Val size: 257


# 4) TF-ID feature extractor 

## Train

In [4]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

#Prueba con solo textos traducidos
#df = df[df["Español"]==False]

In [5]:
from sklearn.preprocessing import LabelEncoder
from models.TIFD import gen_TFID_vectors
from utils.dataset import gen_dataset
import numpy as np

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)
print(X_train.shape, X_test.shape)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


(771, 16725) (193, 16725)


In [ ]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.62, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.61, 'std_test_score': 0.04}
XGBClassifier: {'mean_test_score': 0.59, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.63, 'std_test_score': 0.01}


In [7]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

{'accuracy': 0.6735751295336787, 'precision': 0.7352941176470589, 'recall': 0.6756756756756757, 'f1_macro': 0.6700317512008467, 'cm': array([[55, 27],
       [36, 75]]), 'f1_es': 0.6618249413697328, 'f1_en': 0.6864548494983279, 'cm_es': array([[21, 19],
       [20, 55]]), 'cm_en': array([[34,  8],
       [16, 20]])}
{'accuracy': 0.7046632124352331, 'precision': 0.6901408450704225, 'recall': 0.8828828828828829, 'f1_macro': 0.6730660643704123, 'cm': array([[38, 44],
       [13, 98]]), 'f1_es': 0.6480027056480403, 'f1_en': 0.705565052231719, 'cm_es': array([[ 9, 31],
       [ 3, 72]]), 'cm_en': array([[29, 13],
       [10, 26]])}
{'accuracy': 0.6735751295336787, 'precision': 0.6935483870967742, 'recall': 0.7747747747747747, 'f1_macro': 0.6573481752853318, 'cm': array([[44, 38],
       [25, 86]]), 'f1_es': 0.6371541501976284, 'f1_en': 0.6914899054433937, 'cm_es': array([[13, 27],
       [12, 63]]), 'cm_en': array([[31, 11],
       [13, 23]])}
{'accuracy': 0.689119170984456, 'precision': 0.

## Save

In [ ]:
import mlflow
import git 
#MLflow logging helper function
def safe_log_metric(name, value):
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            if np.size(value) == 1:
                value = float(np.array(value).item())
            else:
                raise ValueError("Métrica con más de un valor.")
        else:
            value = float(value)
        mlflow.log_metric(name, value)
    except Exception as e:
        print(f"⚠️ No se pudo loggear {name}: {e}")

def mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es):
    # Set backend store
    mlflow.set_tracking_uri("http://mlflow-server:5000")
    tracking_uri = mlflow.get_tracking_uri()
    print("Current tracking uri: {}".format(tracking_uri)) 

    # Define el experimento (lo crea si no existe)
    mlflow.set_experiment(exp_info["exp_name"])
    
    # Obtener commit actual
    repo = git.Repo(search_parent_directories=True)
    commit_hash = repo.head.object.hexsha

    for model_name, metrics in results_val.items():
        model = models_dicc[model_name]

        with mlflow.start_run(run_name=model_name):
            print(f"📝 Registrando modelo en MLflow: {model_name}")

            # Hiperparámetros
            try:
                mlflow.log_params(model.get_params())
            except:
                print(f"⚠️ No se pudieron loggear los hiperparámetros para {model_name}")

            #Parámetros adicionales
            for k, v in extra_parms.items():
                mlflow.log_param(k, v)

            # Métricas de validación
            for k, v in metrics.items():
                safe_log_metric(f"val_{k}", v)

            # Métricas de test
            results_test = eval_model(model, X_test, y_test, lang_es)
            for k, v in results_test.items():
                if k.startswith("cm"):
                    # Guardar confusion matrix (o similar) como artefacto
                    # Guardar como CSV temporal
                    fname = f"{k}.csv"
                    np.savetxt(fname, v, delimiter=",", fmt="%d")

                    mlflow.log_artifact(fname, artifact_path="confusion_matrices")

                    # Eliminar archivo local si no lo necesitas
                    os.remove(fname)

                else:
                    # Guardar métrica numérica
                    safe_log_metric(f"test_{k}", v)
            
            #Guardar commit de git
            mlflow.log_param("git_commit", commit_hash)
                    
            # Guardar modelo
            mlflow.sklearn.log_model(model, name = "model", input_example=X_test[:5])

In [11]:
from pipelines.ML_pipeline_skp import mlflow_ckeckpoint

exp_info = {
    'exp_name': "Bayesiansearchcv_TFID_f1w",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

2025/08/26 19:28:25 INFO mlflow.tracking.fluent: Experiment with name 'Bayesiansearchcv_TFID_f1w' does not exist. Creating a new experiment.


Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression
🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/390025476552696805/runs/4bbd9c97fc6e4a4fb39fccdde97664eb
🧪 View experiment at: http://mlflow-server:5000/#/experiments/390025476552696805
📝 Registrando modelo en MLflow: RandomForestClassifier
🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/390025476552696805/runs/fa1efaf491e940d28700478bf84774dc
🧪 View experiment at: http://mlflow-server:5000/#/experiments/390025476552696805
📝 Registrando modelo en MLflow: XGBClassifier
🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/390025476552696805/runs/6edd2213983e4a78a9a9654c72ea9b57
🧪 View experiment at: http://mlflow-server:5000/#/experiments/390025476552696805
📝 Registrando modelo en MLflow: SVC
🏃 View run SVC at: http://mlflow-server:5000/#/experiments/390025476552696805/runs/3aed050be6b6485c8baec77dfe63c276
🧪 View experi

# 5) SPECTER model

## Train

In [8]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [9]:
#del gen_dataset
from utils.dataset import gen_dataset
from models.specter import embed_texts
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# 2) Calcular embeddings
# Parámetros modelo
BASE_MODEL = "allenai/specter2_base"
#ADAPTER_NAME = "allenai/specter2"
ADAPTER_NAME="allenai/specter2_classification"
X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)

print(X_train.shape, X_test.shape)

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

...

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
(771, 768) (193, 768)


In [ ]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.64, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.59, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.61, 'std_test_score': 0.03}
SVC: {'mean_test_score': 0.62, 'std_test_score': 0.0}


In [11]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6476683937823834, 'precision': 0.7087378640776699, 'recall': 0.6576576576576577, 'f1_macro': 0.6434470767224516, 'cm': array([[52, 30],
       [38, 73]]), 'f1_es': 0.6325030804435839, 'f1_en': 0.6575154426904598, 'cm_es': array([[18, 22],
       [20, 55]]), 'cm_en': array([[34,  8],
       [18, 18]])}
RandomForestClassifier
{'accuracy': 0.6373056994818653, 'precision': 0.664, 'recall': 0.7477477477477478, 'f1_macro': 0.6183615819209038, 'cm': array([[40, 42],
       [28, 83]]), 'f1_es': 0.6129706871252124, 'f1_en': 0.6541306794471352, 'cm_es': array([[14, 26],
       [17, 58]]), 'cm_en': array([[26, 16],
       [11, 25]])}
XGBClassifier
{'accuracy': 0.6580310880829016, 'precision': 0.6991150442477876, 'recall': 0.7117117117117117, 'f1_macro': 0.6489748677248677, 'cm': array([[48, 34],
       [32, 79]]), 'f1_es': 0.6475378168742013, 'f1_en': 0.6657807308970101, 'cm_es': array([[18, 22],
       [18, 57]]), 'cm_en': array([[30, 12],
       [14, 22]])}
SVC

## Save

In [16]:
exp_info = {
    'exp_name': "Bayesiansearchcv_specter_f1m",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression
🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/646399578786171018/runs/2196ab9326c24423911cf8e61b308b8a
🧪 View experiment at: http://mlflow-server:5000/#/experiments/646399578786171018
📝 Registrando modelo en MLflow: RandomForestClassifier
🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/646399578786171018/runs/1cf1f20d1ce945a6ba95a883f4d0d05c
🧪 View experiment at: http://mlflow-server:5000/#/experiments/646399578786171018
📝 Registrando modelo en MLflow: XGBClassifier
🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/646399578786171018/runs/bc093ba7b4fd4993b7b9f05a872f7f2e
🧪 View experiment at: http://mlflow-server:5000/#/experiments/646399578786171018
📝 Registrando modelo en MLflow: SVC
🏃 View run SVC at: http://mlflow-server:5000/#/experiments/646399578786171018/runs/1f037a8ef21648b2a3ad0dd4e7ea10e1
🧪 View experi

# 6) ROBERTA

In [17]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [17]:
#del gen_dataset
from utils.dataset import gen_dataset
from models.specter import embed_texts

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [101]:
from transformers import RobertaTokenizerFast, RobertaModel
import torch
import warnings
import tqmd
warnings.filterwarnings("ignore", message="Some weights of the model.*were not initialized.*")


def roberta_encoder(texts):
    embeddings = []
    model_name = "roberta-large"
    tokenizer = RobertaTokenizerFast.from_pretrained(model_name)
    model = RobertaModel.from_pretrained(model_name)
    
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)

        with torch.no_grad():
            outputs = model(**inputs)

        # outputs.last_hidden_state → (batch_size, seq_len, hidden_dim)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # primer token (<s>), equivalente a [CLS]
        embeddings.append(cls_embedding)
    
    return np.array(embeddings)


def roberta_encoder_batch(texts, batch_size=8, max_length=512):
    model_name = "roberta-large"
    tokenizer = RobertaTokenizerFast.from_pretrained(model_name)
    model = RobertaModel.from_pretrained(model_name)

    model.eval()  # desactiva dropout
    embeddings = []

    # recorrer en lotes de batch_size
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        # tokenización por lote
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        with torch.no_grad():
            outputs = model(**inputs)

        # embeddings del token <s> ([CLS]) para cada texto del batch
        cls_embeddings = outputs.last_hidden_state[:, 0, :]  # (batch, hidden_dim)
        embeddings.append(cls_embeddings.cpu().numpy())

    # concatenar todos los batches
    return np.vstack(embeddings)  # (n_texts, hidden_dim)

ModuleNotFoundError: No module named 'tqmd'

In [100]:
X_train = roberta_encoder_batch(X_train)
X_test= roberta_encoder_batch(X_test)

KeyboardInterrupt: 